# Process Mining — Group 5

Discovery, evaluation and performance analysis of the BPI Challenge 2020 **Travel Permit** log
(`PermitLog.xes`), scoped per object flow (Permit / Declaration / Request For Payment).


## Setup


In [1]:
import os
import re
import pandas as pd
import matplotlib
import pm4py
from pm4py.algo.filtering.dfg import dfg_filtering
from pm4py.visualization.dfg import visualizer as dfg_visualizer
from pm4py.visualization.dfg.variants.frequency import Parameters as DFGParam
from pm4py.visualization.petri_net import visualizer as pn_visualizer
from pm4py.visualization.petri_net.common.visualize import Parameters as PNParam

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"
LIFECYCLE = "lifecycle:transition"

XES_PATH = "PermitLog.xes"
OUTPUT_FOLDER = "Output"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

MIN_ACTIVITY_FREQ = 0.01
VARIANT_COVERAGE = 0.02
SUBPROCESSES = {
    "permit":              {"prefixes": ["Permit"],              "extra": ["Start trip", "End trip"]},
    "declaration":         {"prefixes": ["Declaration"],         "extra": ["Send Reminder"]},
    "request_for_payment": {"prefixes": ["Request For Payment"], "extra": ["Request Payment", "Payment Handled"]},
}
DISCOVERY_SCOPES = dict(SUBPROCESSES)
BASE_SAMPLE_SIZES = [250, 500, 1000]
heuristics_settings = [
    {"dependency_threshold": 0.20, "and_threshold": 0.40, "loop_two_threshold": 0.20},
    {"dependency_threshold": 0.30, "and_threshold": 0.50, "loop_two_threshold": 0.30},
    {"dependency_threshold": 0.40, "and_threshold": 0.60, "loop_two_threshold": 0.40},
]
inductive_settings = [{"noise_threshold": t} for t in (0.00, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40)]
DECISION_METRICS = ["fitness_log", "precision", "generalization", "simplicity"]

def load_event_log():
    """Read the XES log and return a clean event DataFrame: complete events only,
    projected to [case, activity, timestamp] with parsed UTC timestamps."""
    df = pm4py.convert_to_dataframe(pm4py.read_xes(XES_PATH))
    if LIFECYCLE in df.columns:
        complete = df[LIFECYCLE].astype(str).str.lower().eq("complete")
        if complete.any():
            df = df[complete]
    df = df[[CASE_ID, ACTIVITY, TIMESTAMP]].dropna().copy()
    df[TIMESTAMP] = pd.to_datetime(df[TIMESTAMP], errors="coerce", utc=True)
    df = df.dropna(subset=[TIMESTAMP])
    return df.reset_index(drop=True)

EVENT_LOG = load_event_log()


c:\Users\tomwo\Documents\GitHub\process-mining-group5\venv\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
c:\Users\tomwo\Documents\GitHub\process-mining-group5\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 7065/7065 [00:01<00:00, 5537.56it/s]


## Directly-Follows Graph (overview)

A DFG of the full event log (`complete` events only) to get an overview of the process before
scoping and mining the individual sub-processes. **Node colour shows frequency** (darker = more
frequent), so the colour gradient highlights the main path through the flow. To keep colour free
for that purpose, the **sub-processes are distinguished structurally instead**: each flow's
activities are wrapped in a labelled, dashed box (Permit / Declaration / Request For Payment).


In [2]:
dfg_log = EVENT_LOG.sort_values([CASE_ID, TIMESTAMP], kind="mergesort").reset_index(drop=True)

dfg, start_activities, end_activities = pm4py.discover_dfg(
    dfg_log, activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
)

activities_count = dfg_log[ACTIVITY].value_counts().to_dict()
dfg, start_activities, end_activities, activities_count = dfg_filtering.filter_dfg_on_activities_percentage(
    dfg, start_activities, end_activities, activities_count, 0.30,
)
dfg, start_activities, end_activities, activities_count = dfg_filtering.filter_dfg_on_paths_percentage(
    dfg, start_activities, end_activities, activities_count, 0.175,
)

dfg_gviz = dfg_visualizer.apply(
    dfg,
    variant=dfg_visualizer.Variants.FREQUENCY,
    parameters={
        DFGParam.FORMAT: "png",
        DFGParam.START_ACTIVITIES: start_activities,
        DFGParam.END_ACTIVITIES: end_activities,
        DFGParam.FONT_SIZE: 20,
    },
)


def _activity_subprocess(act):
    for sp_name, sp in SUBPROCESSES.items():
        if act.startswith(tuple(sp["prefixes"])) or act in sp.get("extra", []):
            return sp_name
    return None

node_subprocess = {}
for act in ({a for edge in dfg for a in edge} | set(start_activities) | set(end_activities)):
    sp_name = _activity_subprocess(act)
    if sp_name is not None:
        node_subprocess[str(hash(act))] = sp_name

cluster_nodes = {sp_name: [] for sp_name in SUBPROCESSES}
other_lines = []
insert_at = None
for _line in dfg_gviz.body:
    _m = re.match(r'\s*"?(-?\d+)"?\s+\[', _line)
    if _m and _m.group(1) in node_subprocess:
        if insert_at is None:
            insert_at = len(other_lines)
        cluster_nodes[node_subprocess[_m.group(1)]].append(_line.strip())
        continue
    other_lines.append(_line)

cluster_block = []
for _i, sp_name in enumerate(SUBPROCESSES):
    _lines = cluster_nodes[sp_name]
    if not _lines:
        continue
    cluster_block.append(f"subgraph cluster_{_i} {{")
    cluster_block.append(f'\tlabel="{sp_name.replace("_", " ").title()}"')
    cluster_block.append('\tstyle="rounded,dashed"')
    cluster_block.append('\tcolor="#666666"')
    cluster_block.append("\tpenwidth=2")
    cluster_block.append("\tfontsize=30")
    cluster_block.append('\tfontname="Helvetica-Bold"')
    cluster_block.append("\tlabeljust=l")
    cluster_block.extend("\t" + _l for _l in _lines)
    cluster_block.append("}")

if insert_at is None:
    insert_at = len(other_lines)
other_lines[insert_at:insert_at] = cluster_block
dfg_gviz.body = other_lines

dfg_gviz.graph_attr["rankdir"] = "TB"
for _attr in (dfg_gviz.graph_attr, dfg_gviz.node_attr, dfg_gviz.edge_attr):
    _attr["fontname"] = "Helvetica-Bold"
dfg_visualizer.save(dfg_gviz, os.path.join(OUTPUT_FOLDER, "permitlog_dfg.png"))


''

## Preprocessing & data quality

Applied in the discovery cell before mining:

1. Keep only `complete` lifecycle events.
2. Drop exact duplicate `(case, activity, timestamp)` rows.
3. Stable event ordering: events are sorted by `(case, timestamp)` with a **stable** sort, so when two events share a timestamp the log's native recording order is kept rather than re-ordered alphabetically. This matters for planned-date activities such as `Start trip` / `End trip`, which share an identical timestamp in ~500 cases: an alphabetical tie-break would place `End trip` before `Start trip` and invent a false `End trip → Start trip` edge, which makes the Inductive Miner treat the two as an exclusive choice instead of a sequence. The **original** timestamps are kept throughout, so the mined ordering reflects the real event times.
4. Scope mining per object flow (Permit, Declaration, Request For Payment) to avoid a spaghetti model. Activities that have no flow-specific name prefix are attached to the flow they belong to: `Start trip` / `End trip` join the Permit flow (a trip is authorised by a permit), `Request Payment` / `Payment Handled` join the Request For Payment flow (they settle a request for payment), and `Send Reminder` joins the Declaration flow (it nudges the employee to submit the declaration — it is preceded by `End trip` and followed by `Declaration SUBMITTED`). This assigns every one of the log's 51 activities to exactly one sub-process.
5. Rare-activity filter (`MIN_ACTIVITY_FREQ`): within each sub-process, drop activities occurring in fewer than 1% of that flow's cases (set to `0` to disable).
6. Self-loop collapsing: within each case, consecutive repeats of the same activity (`A, A → A`) are merged into a single event. Such immediate repetitions create an `A → A` edge in the directly-follows graph, which the Inductive Miner can only express as a redo loop (`A` plus a silent `τ` back-edge) and which IMf's `noise_threshold` rarely filters out. Collapsing them removes those spurious self-loops.
7. Trace-variant coverage filter (`VARIANT_COVERAGE`): applied just before mining. During tuning it is applied to each candidate sample; during final model creation it is applied to the full subprocess log, so it no longer limits the final Inductive Miner model to a 1,000-case sample. The rare survivors of a process are mostly rejection/rework variants; the Inductive Miner can only fit them by wrapping a large subtree in a loop or choice that then permits far more behaviour than the log ever shows, which is the main drag on precision. Filtering on whole variants (rather than `noise_threshold`, which prunes individual edges and can cut mid-sequence) removes that behaviour cleanly while keeping the mainstream flow intact (set to `0` to disable).



## Discovery & evaluation

- **Miners:** Heuristics and Inductive.
- The Inductive Miner always returns a sound, perfectly-replayable net, so its quality ceiling is **precision** (and simplicity), not fitness. Two levers target that directly: a wider `noise_threshold` sweep (`0.0 … 0.40`) that prunes infrequent directly-follows edges, and variant-coverage filtering (`VARIANT_COVERAGE`) that drops rare rejection/rework paths before mining so they are not wrapped in over-permissive loop blocks.
- **Metrics:** fitness, precision, generalization and simplicity — fitness and precision use alignment-based measures (accurate but slower) — plus their F-score and an averaged score, which drives the model selection.
- The tuning grid can still compare 250/500/1,000-case samples and the full subprocess log. After the best settings are chosen, the final PNML/PNG is always re-mined on the full subprocess log using those winning settings. This fixes the Declaration + Inductive Miner case where the best settings could come from 1,000 sampled cases but the final model should use all Declaration cases.


In [ ]:
def reduce_net(net, im, fm):
    """Simplify a discovered net by removing silent transitions and implicit places."""
    net = pm4py.reduce_petri_net_invisibles(net)
    net, im, fm = pm4py.reduce_petri_net_implicit_places(net, im, fm)
    return net, im, fm

def evaluate_net(net, im, fm, eval_df):
    """Alignment-based quality metrics for a net, plus F-score and an averaged score."""
    fit = pm4py.fitness_alignments(
        eval_df, net, im, fm,
        activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
    )
    precision = pm4py.precision_alignments(
        eval_df, net, im, fm,
        activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
    )
    metrics = {
        "fitness_log": fit.get("log_fitness"),
        "precision": precision,
        "generalization": pm4py.generalization_tbr(
            eval_df, net, im, fm,
            activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
        ),
        "simplicity": pm4py.simplicity_petri_net(net, im, fm),
        "f_score": None,
        "avg_score": None,
    }
    f, p = metrics["fitness_log"], metrics["precision"]
    if f is not None and p is not None and (f + p) > 0:
        metrics["f_score"] = 2 * f * p / (f + p)
    available = [metrics[k] for k in DECISION_METRICS if metrics[k] is not None]
    if available:
        metrics["avg_score"] = round(sum(available) / len(available), 6)
    return metrics

df = EVENT_LOG.drop_duplicates(subset=[CASE_ID, ACTIVITY, TIMESTAMP]).copy()


def prepare_subprocess_discovery_df(sp):
    """Return the full, preprocessed event log for one subprocess.

    This function intentionally does not sample cases. It is reused both by the
    tuning grid and by the final all-case mining step, so the winning settings are
    applied to the same full subprocess log rather than to the sample that found them.
    """
    prefix_mask = df[ACTIVITY].str.startswith(tuple(sp["prefixes"]))
    extra_mask = df[ACTIVITY].isin(sp.get("extra", []))
    sp_df = df[prefix_mask | extra_mask].sort_values(
        [CASE_ID, TIMESTAMP], kind="mergesort"
    ).reset_index(drop=True)
    if sp_df.empty:
        return sp_df

    if MIN_ACTIVITY_FREQ > 0:
        cases_per_activity = sp_df.groupby(ACTIVITY)[CASE_ID].nunique()
        min_cases = MIN_ACTIVITY_FREQ * sp_df[CASE_ID].nunique()
        keep = cases_per_activity[cases_per_activity >= min_cases].index
        sp_df = sp_df[sp_df[ACTIVITY].isin(keep)].reset_index(drop=True)
    if sp_df.empty:
        return sp_df

    same_as_prev = (
        sp_df[CASE_ID].eq(sp_df[CASE_ID].shift())
        & sp_df[ACTIVITY].eq(sp_df[ACTIVITY].shift())
    )
    return sp_df[~same_as_prev].reset_index(drop=True)


def apply_variant_coverage_filter(log_df):
    """Apply the optional variant-coverage filter to the given log.

    When this is called from final mining, it is applied to the full subprocess log,
    not to the 1,000-case tuning sample.
    """
    if VARIANT_COVERAGE <= 0:
        return log_df.sort_values([CASE_ID, TIMESTAMP], kind="mergesort").reset_index(drop=True)

    filtered = pm4py.filter_variants_by_coverage_percentage(
        log_df, VARIANT_COVERAGE,
        activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
    )
    return filtered.sort_values([CASE_ID, TIMESTAMP], kind="mergesort").reset_index(drop=True)


results = []

for sp_name, sp in DISCOVERY_SCOPES.items():
    sp_df = prepare_subprocess_discovery_df(sp)
    if sp_df.empty:
        continue

    all_cases = pd.Series(sp_df[CASE_ID].unique()).sample(frac=1, random_state=42).tolist()
    case_sample_sizes = [x for x in BASE_SAMPLE_SIZES if x < len(all_cases)] + [len(all_cases)]

    for sample_size in case_sample_sizes:
        selected = all_cases[:sample_size]
        eval_df = sp_df[sp_df[CASE_ID].isin(selected)].sort_values(
            [CASE_ID, TIMESTAMP], kind="mergesort"
        ).reset_index(drop=True)

        eval_df = apply_variant_coverage_filter(eval_df)
        if eval_df.empty:
            continue

        for setting in heuristics_settings:
            label = (
                f"{sp_name}_heuristics_cases_{sample_size}"
                f"_dep_{str(setting['dependency_threshold']).replace('.', '_')}"
                f"_and_{str(setting['and_threshold']).replace('.', '_')}"
                f"_loop2_{str(setting['loop_two_threshold']).replace('.', '_')}"
            )
            net, im, fm = reduce_net(*pm4py.discover_petri_net_heuristics(eval_df, **setting))
            pm4py.write_pnml(net, im, fm, os.path.join(OUTPUT_FOLDER, label + ".pnml"))
            results.append({
                "subprocess": sp_name,
                "algorithm": "Heuristics Miner",
                "sample_cases": sample_size,
                "sample_events": len(eval_df),
                "dependency_threshold": setting["dependency_threshold"],
                "and_threshold": setting["and_threshold"],
                "loop_two_threshold": setting["loop_two_threshold"],
                "noise_threshold": None,
                **evaluate_net(net, im, fm, eval_df),
            })

        for setting in inductive_settings:
            label = (
                f"{sp_name}_inductive_cases_{sample_size}"
                f"_noise_{str(setting['noise_threshold']).replace('.', '_')}"
            )
            net, im, fm = reduce_net(*pm4py.discover_petri_net_inductive(eval_df, **setting))
            pm4py.write_pnml(net, im, fm, os.path.join(OUTPUT_FOLDER, label + ".pnml"))
            results.append({
                "subprocess": sp_name,
                "algorithm": "Inductive Miner",
                "sample_cases": sample_size,
                "sample_events": len(eval_df),
                "dependency_threshold": None,
                "and_threshold": None,
                "loop_two_threshold": None,
                "noise_threshold": setting["noise_threshold"],
                **evaluate_net(net, im, fm, eval_df),
            })

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUTPUT_FOLDER, "permitlog_mining_quality_results.csv"), index=False)
results_df


aligning log, completed variants :: 100%|██████████| 7/7 [00:00<00:00, 720.64it/s]
computing precision with alignments, completed variants :: 100%|██████████| 22/22 [00:00<00:00, 3126.37it/s]
aligning log, completed variants :: 100%|██████████| 7/7 [00:00<00:00, 819.13it/s]
computing precision with alignments, completed variants :: 100%|██████████| 22/22 [00:00<00:00, 2660.75it/s]
aligning log, completed variants :: 100%|██████████| 7/7 [00:00<00:00, 817.01it/s]
computing precision with alignments, completed variants :: 100%|██████████| 22/22 [00:00<00:00, 3149.74it/s]
aligning log, completed variants :: 100%|██████████| 7/7 [00:00<00:00, 723.05it/s]
computing precision with alignments, completed variants :: 100%|██████████| 22/22 [00:00<00:00, 2689.52it/s]
aligning log, completed variants :: 100%|██████████| 7/7 [00:00<00:00, 751.38it/s]
computing precision with alignments, completed variants :: 100%|██████████| 22/22 [00:00<00:00, 2815.06it/s]
aligning log, completed variants :: 100%

,subprocess,algorithm,sample_cases,sample_events,dependency_threshold,and_threshold,loop_two_threshold,noise_threshold,fitness_log,precision,generalization,simplicity,f_score,avg_score_percent
0,permit,Heuristics Miner,250,1051,0.2,0.4,0.2,NaN,0.923062,0.985536,0.848352,0.666667,0.953277,0.855904
1,permit,Heuristics Miner,250,1051,0.3,0.5,0.3,NaN,0.923062,0.985536,0.848352,0.666667,0.953277,0.855904
2,permit,Heuristics Miner,250,1051,0.4,0.6,0.4,NaN,0.873665,0.981340,0.793590,0.674419,0.924378,0.830754
3,permit,Inductive Miner,250,1051,NaN,NaN,NaN,0.00,0.999949,0.449772,0.883794,0.714286,0.620463,0.761950
4,permit,Inductive Miner,250,1051,NaN,NaN,NaN,0.05,0.999949,0.449772,0.883794,0.714286,0.620463,0.761950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,request_for_payment,Inductive Miner,5725,12649,NaN,NaN,NaN,0.10,0.999933,1.000000,0.966374,0.833333,0.999967,0.949910
116,request_for_payment,Inductive Miner,5725,12649,NaN,NaN,NaN,0.15,0.975961,0.969319,0.868753,0.818182,0.972628,0.908053
117,request_for_payment,Inductive Miner,5725,12649,NaN,NaN,NaN,0.20,0.975961,0.969319,0.868753,0.818182,0.972628,0.908053
118,request_for_payment,Inductive Miner,5725,12649,NaN,NaN,NaN,0.30,0.975961,0.969319,0.868753,0.818182,0.972628,0.908053


## Performance overlay (green → red throughput)

For each best model we overlay the **throughput time** on every activity block and colour it
from **green (fastest)** to **red (slowest)** with the `RdYlGn_r` colormap.

- The throughput of a block is the waiting time leading **into** that activity (time since the
  previous step of the same flow). We report both the **mean** and the **median** over all cases,
  because this log is heavily skewed: a few outlier waits pull the mean up while most cases are fast.
- **Colour blends both statistics.** Mean and median are normalised on a shared scale and averaged,
  so a step only turns deep red when *both* are high (a genuine, consistent bottleneck) rather than
  when a handful of outliers inflate the mean alone. The label prints `mean … | med …`.
- pm4py has no built-in green→red Petri-net heatmap, so we feed a custom `decorations` dict
  (per-transition `label` + `color`) straight to the renderer.
- Times are measured on the **original** timestamps read straight from the log.
- *Caveat:* `Start trip` / `End trip` / `Payment Handled` carry planned dates that can precede the
  prior step, so negative gaps are clipped to 0 and their times are noisier than the permit steps.


In [ ]:
scored = results_df.dropna(subset=["avg_score"])
best_per_algo = scored.loc[
    scored.groupby(["subprocess", "algorithm"])["avg_score"].idxmax()
].sort_values(["subprocess", "algorithm"])

raw = EVENT_LOG.sort_values([CASE_ID, TIMESTAMP], kind="mergesort").reset_index(drop=True)
CMAP = matplotlib.colormaps["RdYlGn_r"]


def human_time(seconds):
    s = float(seconds)
    for unit, size in (("d", 86400.0), ("h", 3600.0), ("m", 60.0)):
        if s >= size:
            return f"{s / size:.1f}{unit}"
    return f"{s:.0f}s"


def activity_throughput(prefixes, extra):
    """Mean & median waiting time into each activity of a flow (seconds, negatives clipped)."""
    mask = raw[ACTIVITY].str.startswith(tuple(prefixes)) | raw[ACTIVITY].isin(extra)
    sub = raw[mask].copy()
    sub["_delta"] = sub.groupby(CASE_ID)[TIMESTAMP].diff().dt.total_seconds().clip(lower=0.0)
    return sub.dropna(subset=["_delta"]).groupby(ACTIVITY)["_delta"].agg(mean="mean", median="median")


def final_model_label(row, final_case_count):
    """Label for the final model mined with the winning settings on all available cases."""
    scope = row["subprocess"]
    if row["algorithm"] == "Heuristics Miner":
        return (f"{scope}_heuristics_all_cases_{final_case_count}"
                f"_dep_{str(float(row['dependency_threshold'])).replace('.', '_')}"
                f"_and_{str(float(row['and_threshold'])).replace('.', '_')}"
                f"_loop2_{str(float(row['loop_two_threshold'])).replace('.', '_')}")
    return (f"{scope}_inductive_all_cases_{final_case_count}"
            f"_noise_{str(float(row['noise_threshold'])).replace('.', '_')}")


def discover_final_model_on_full_log(row, mining_df):
    """Mine the full subprocess log with the settings selected during tuning."""
    if row["algorithm"] == "Heuristics Miner":
        return reduce_net(*pm4py.discover_petri_net_heuristics(
            mining_df,
            dependency_threshold=float(row["dependency_threshold"]),
            and_threshold=float(row["and_threshold"]),
            loop_two_threshold=float(row["loop_two_threshold"]),
        ))

    return reduce_net(*pm4py.discover_petri_net_inductive(
        mining_df,
        noise_threshold=float(row["noise_threshold"]),
    ))


final_rows = []

for _, row in best_per_algo.iterrows():
    sp = DISCOVERY_SCOPES[row["subprocess"]]
    full_df = prepare_subprocess_discovery_df(sp)
    if full_df.empty:
        continue
    
    mining_df = apply_variant_coverage_filter(full_df)
    if mining_df.empty:
        continue

    final_case_count = mining_df[CASE_ID].nunique()
    label = final_model_label(row, final_case_count)
    net, im, fm = discover_final_model_on_full_log(row, mining_df)
    pm4py.write_pnml(net, im, fm, os.path.join(OUTPUT_FOLDER, label + ".pnml"))

    final_rows.append({
        "subprocess": row["subprocess"],
        "algorithm": row["algorithm"],
        "tuned_on_sample_cases": int(row["sample_cases"]),
        "final_mined_cases": final_case_count,
        "final_mined_events": len(mining_df),
        "final_model_label": label,
        "dependency_threshold": row["dependency_threshold"],
        "and_threshold": row["and_threshold"],
        "loop_two_threshold": row["loop_two_threshold"],
        "noise_threshold": row["noise_threshold"],
        **evaluate_net(net, im, fm, mining_df),
    })

    thr = activity_throughput(sp["prefixes"], sp.get("extra", []))
    if thr.empty:
        continue

    both = pd.concat([thr["mean"], thr["median"]])
    vmin, vmax = float(both.min()), float(both.max())
    norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax if vmax > vmin else vmin + 1.0)

    decorations = {}
    for t in net.transitions:
        if t.label is None:
            continue
        if t.label in thr.index:
            mean_val = float(thr.loc[t.label, "mean"])
            median_val = float(thr.loc[t.label, "median"])
            combined = float((norm(mean_val) + norm(median_val)) / 2.0)
            decorations[t] = {
                "label": f"{t.label}\nmean {human_time(mean_val)} | med {human_time(median_val)}",
                "color": matplotlib.colors.to_hex(CMAP(combined)),
            }
        else:
            decorations[t] = {
                "label": f"{t.label}\n(start)",
                "color": matplotlib.colors.to_hex(CMAP(0.0)),
            }

    gviz = pn_visualizer.apply(
        net, im, fm,
        variant=pn_visualizer.Variants.WO_DECORATION,
        parameters={
            PNParam.FORMAT: "png",
            PNParam.DECORATIONS: decorations,
            PNParam.RANKDIR: "TB",
            PNParam.FONT_SIZE: 20,
        },
    )
    for _attr in (gviz.graph_attr, gviz.node_attr, gviz.edge_attr):
        _attr["fontname"] = "Helvetica-Bold"
    pn_visualizer.save(gviz, os.path.join(OUTPUT_FOLDER, label + ".png"))

FINAL_TABLE_COLUMNS_TO_REMOVE = [
    "tuned_on_sample_cases",
    "final_mined_cases",
    "final_mined_events",
    "final_model_label",
]

final_models_df = pd.DataFrame(final_rows).drop(columns=FINAL_TABLE_COLUMNS_TO_REMOVE, errors="ignore")
final_models_df.to_csv(os.path.join(OUTPUT_FOLDER, "permitlog_final_all_case_models.csv"), index=False)
best_per_algo.to_csv(os.path.join(OUTPUT_FOLDER, "permitlog_best_per_algo.csv"), index=False)
final_models_df


aligning log, completed variants :: 100%|██████████| 8/8 [00:00<00:00, 882.59it/s]
computing precision with alignments, completed variants :: 100%|██████████| 12/12 [00:00<00:00, 3561.03it/s]
aligning log, completed variants :: 100%|██████████| 8/8 [00:00<00:00, 1048.48it/s]
computing precision with alignments, completed variants :: 100%|██████████| 12/12 [00:00<00:00, 4342.30it/s]
aligning log, completed variants :: 100%|██████████| 7/7 [00:00<00:00, 778.85it/s]
computing precision with alignments, completed variants :: 100%|██████████| 22/22 [00:00<00:00, 2957.43it/s]
aligning log, completed variants :: 100%|██████████| 7/7 [00:00<00:00, 1008.32it/s]
computing precision with alignments, completed variants :: 100%|██████████| 22/22 [00:00<00:00, 4887.17it/s]
aligning log, completed variants :: 100%|██████████| 3/3 [00:00<00:00, 794.53it/s]
computing precision with alignments, completed variants :: 100%|██████████| 12/12 [00:00<00:00, 4660.77it/s]
aligning log, completed variants :: 10

,subprocess,algorithm,dependency_threshold,and_threshold,loop_two_threshold,noise_threshold,fitness_log,precision,generalization,simplicity,f_score,avg_score_percent
0,declaration,Heuristics Miner,0.3,0.5,0.3,NaN,0.925285,0.947841,0.969960,0.696970,0.936427,0.885014
1,declaration,Inductive Miner,NaN,NaN,NaN,0.15,0.972782,0.947841,0.974843,0.733333,0.960150,0.907200
2,permit,Heuristics Miner,0.3,0.5,0.3,NaN,0.918934,0.988143,0.970539,0.695652,0.952283,0.893317
3,permit,Inductive Miner,NaN,NaN,NaN,0.15,0.962487,0.869631,0.974032,0.818182,0.913706,0.906083
4,request_for_payment,Heuristics Miner,0.2,0.4,0.2,NaN,0.951989,1.000000,0.962729,0.882353,0.975404,0.949268
5,request_for_payment,Inductive Miner,NaN,NaN,NaN,0.00,0.999933,1.000000,0.966374,0.833333,0.999967,0.949910


## Bottleneck statistics

- Waiting time
- Activities that take the longest
- Send reminder time
- How long between end of trip and submission of declaration
- Average lenght of each subprocess
- Amount of identified bottlenecks in subprocess

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 40)  # truncate long strings
pd.set_option('display.width', 120)         # wider output

In [ ]:
for name, i in zip(['Complete process', 'Permit application', 'Declaration', 'Request for payment'], df):
    total_cases = i['case:concept:name'].nunique()
    reminder_cases = i[i['concept:name'] == 'Send Reminder']['case:concept:name'].nunique()
    reminder_pct = reminder_cases / total_cases * 100
    
    print(f"{name}: {reminder_cases} cases ({reminder_pct:.2f}%)")

In [ ]:
i = df[0].sort_values(['case:concept:name', 'time:timestamp'])

results = []
stopped_at_reminder = []

for case_id, case_df in i.groupby('case:concept:name'):
    activities = case_df['concept:name'].tolist()
    timestamps = case_df['time:timestamp'].tolist()
    
    j = 0
    while j < len(activities):
        if activities[j] == 'End trip':
            k = j + 1
            reminder_count = 0
            reminder_timestamps = []
            
            while k < len(activities) and activities[k] == 'Send Reminder':
                reminder_count += 1
                reminder_timestamps.append(timestamps[k])
                k += 1
            
            if reminder_count > 0:
                end_trip_time = timestamps[j]
                wait_to_first_reminder = (reminder_timestamps[0] - end_trip_time).total_seconds() / 3600 / 24
                
                # time between each reminder
                between_reminders = []
                for r in range(1, len(reminder_timestamps)):
                    gap = (reminder_timestamps[r] - reminder_timestamps[r-1]).total_seconds() / 3600 / 24
                    between_reminders.append(gap)

                # sequence continues to declaration
                if k < len(activities) and activities[k] == 'Declaration SUBMITTED by EMPLOYEE':
                    declaration_time = timestamps[k]
                    total_time = (declaration_time - end_trip_time).total_seconds() / 3600 / 24
                    wait_after_last_reminder = (declaration_time - reminder_timestamps[-1]).total_seconds() / 3600 / 24
                    
                    results.append({
                        'case': case_id,
                        'reminder_count': reminder_count,
                        'total_time_hrs': total_time,
                        'wait_end_trip_to_first_reminder_hrs': wait_to_first_reminder,
                        'between_reminders': between_reminders,
                        'wait_last_reminder_to_declaration_hrs': wait_after_last_reminder
                    })
                
                # case stops at send reminder
                elif k == len(activities):
                    time_to_last_reminder = (reminder_timestamps[-1] - end_trip_time).total_seconds() / 3600 / 24
                    
                    stopped_at_reminder.append({
                        'case': case_id,
                        'reminder_count': reminder_count,
                        'time_end_trip_to_last_reminder_hrs': time_to_last_reminder,
                        'wait_end_trip_to_first_reminder_hrs': wait_to_first_reminder,
                        'between_reminders': between_reminders
                    })
            j = k
        else:
            j += 1

results_df = pd.DataFrame(results)
stopped_df = pd.DataFrame(stopped_at_reminder)

def print_between_reminders(group, reminder_count):
    if reminder_count > 1:
        all_gaps = [gap for gaps in group['between_reminders'] for gap in gaps]
        mean_gap = sum(all_gaps) / len(all_gaps)
        median_gap = pd.Series(all_gaps).median()
        print(f"    Between reminders   — Mean: {mean_gap:.2f} days  |  Median: {median_gap:.2f} days")

# === Completed sequences ===
print("=== End Trip → Send Reminder(s) → Declaration SUBMITTED by EMPLOYEE ===")
print(f"\nTotal occurrences: {len(results_df)}")
print(f"Cases involved:    {results_df['case'].nunique()}")

print("\n--- By number of reminders ---")
for n, group in results_df.groupby('reminder_count'):
    print(f"\n  Send Reminder x{n} ({len(group)} occurrences):")
    print(f"    Total time          — Mean: {group['total_time_hrs'].mean():.2f} days  |  Median: {group['total_time_hrs'].median():.2f} days")
    print(f"    End Trip → Reminder — Mean: {group['wait_end_trip_to_first_reminder_hrs'].mean():.2f} days  |  Median: {group['wait_end_trip_to_first_reminder_hrs'].median():.2f} days")
    print_between_reminders(group, n)
    print(f"    Reminder → Decl.    — Mean: {group['wait_last_reminder_to_declaration_hrs'].mean():.2f} days  |  Median: {group['wait_last_reminder_to_declaration_hrs'].median():.2f} days")

# === Stopped at Send Reminder ===
print("\n=== End Trip → Send Reminder(s) → [STOPPED] ===")
print(f"\nTotal occurrences: {len(stopped_df)}")
print(f"Cases involved:    {stopped_df['case'].nunique()}")

if len(stopped_df) > 0:
    print("\n--- By number of reminders ---")
    for n, group in stopped_df.groupby('reminder_count'):
        print(f"\n  Send Reminder x{n} ({len(group)} occurrences):")
        print(f"    End Trip → First Reminder — Mean: {group['wait_end_trip_to_first_reminder_hrs'].mean():.2f} days  |  Median: {group['wait_end_trip_to_first_reminder_hrs'].median():.2f} days")
        print_between_reminders(group, n)
        print(f"    End Trip → Last Reminder  — Mean: {group['time_end_trip_to_last_reminder_hrs'].mean():.2f} days  |  Median: {group['time_end_trip_to_last_reminder_hrs'].median():.2f} days")

In [ ]:
i = df[0].sort_values(['case:concept:name', 'time:timestamp'])

results = []
stopped_at_reminder = []

for case_id, case_df in i.groupby('case:concept:name'):
    activities = case_df['concept:name'].tolist()
    timestamps = case_df['time:timestamp'].tolist()
    
    j = 0
    while j < len(activities):
        if activities[j] == 'End trip':
            k = j + 1
            reminder_count = 0
            reminder_timestamps = []
            
            while k < len(activities) and activities[k] == 'Send Reminder':
                reminder_count += 1
                reminder_timestamps.append(timestamps[k])
                k += 1
            
            if reminder_count > 0:
                end_trip_time = timestamps[j]
                wait_to_first_reminder = (reminder_timestamps[0] - end_trip_time).total_seconds() / 3600 / 24
                
                # time between each reminder
                between_reminders = []
                for r in range(1, len(reminder_timestamps)):
                    gap = (reminder_timestamps[r] - reminder_timestamps[r-1]).total_seconds() / 3600 / 24
                    between_reminders.append(gap)

                # sequence continues to declaration
                if k < len(activities) and activities[k] == 'Declaration SAVED by EMPLOYEE':
                    declaration_time = timestamps[k]
                    total_time = (declaration_time - end_trip_time).total_seconds() / 3600 / 24
                    wait_after_last_reminder = (declaration_time - reminder_timestamps[-1]).total_seconds() / 3600 / 24
                    
                    results.append({
                        'case': case_id,
                        'reminder_count': reminder_count,
                        'total_time_hrs': total_time,
                        'wait_end_trip_to_first_reminder_hrs': wait_to_first_reminder,
                        'between_reminders': between_reminders,
                        'wait_last_reminder_to_declaration_hrs': wait_after_last_reminder
                    })
                
                # case stops at send reminder
                elif k == len(activities):
                    time_to_last_reminder = (reminder_timestamps[-1] - end_trip_time).total_seconds() / 3600 / 24
                    
                    stopped_at_reminder.append({
                        'case': case_id,
                        'reminder_count': reminder_count,
                        'time_end_trip_to_last_reminder_hrs': time_to_last_reminder,
                        'wait_end_trip_to_first_reminder_hrs': wait_to_first_reminder,
                        'between_reminders': between_reminders
                    })
            j = k
        else:
            j += 1

results_df = pd.DataFrame(results)
stopped_df = pd.DataFrame(stopped_at_reminder)

def print_between_reminders(group, reminder_count):
    if reminder_count > 1:
        all_gaps = [gap for gaps in group['between_reminders'] for gap in gaps]
        mean_gap = sum(all_gaps) / len(all_gaps)
        median_gap = pd.Series(all_gaps).median()
        print(f"    Between reminders   — Mean: {mean_gap:.2f} days  |  Median: {median_gap:.2f} days")

# === Completed sequences ===
print("=== End Trip → Send Reminder(s) → Declaration SAVED by EMPLOYEE ===")
print(f"\nTotal occurrences: {len(results_df)}")
print(f"Cases involved:    {results_df['case'].nunique()}")

print("\n--- By number of reminders ---")
for n, group in results_df.groupby('reminder_count'):
    print(f"\n  Send Reminder x{n} ({len(group)} occurrences):")
    print(f"    Total time          — Mean: {group['total_time_hrs'].mean():.2f} days  |  Median: {group['total_time_hrs'].median():.2f} days")
    print(f"    End Trip → Reminder — Mean: {group['wait_end_trip_to_first_reminder_hrs'].mean():.2f} days  |  Median: {group['wait_end_trip_to_first_reminder_hrs'].median():.2f} days")
    print_between_reminders(group, n)
    print(f"    Reminder → Decl.    — Mean: {group['wait_last_reminder_to_declaration_hrs'].mean():.2f} days  |  Median: {group['wait_last_reminder_to_declaration_hrs'].median():.2f} days")

# === Stopped at Send Reminder ===
print("\n=== End Trip → Send Reminder(s) → [STOPPED] ===")
print(f"\nTotal occurrences: {len(stopped_df)}")
print(f"Cases involved:    {stopped_df['case'].nunique()}")

if len(stopped_df) > 0:
    print("\n--- By number of reminders ---")
    for n, group in stopped_df.groupby('reminder_count'):
        print(f"\n  Send Reminder x{n} ({len(group)} occurrences):")
        print(f"    End Trip → First Reminder — Mean: {group['wait_end_trip_to_first_reminder_hrs'].mean():.2f} days  |  Median: {group['wait_end_trip_to_first_reminder_hrs'].median():.2f} days")
        print_between_reminders(group, n)
        print(f"    End Trip → Last Reminder  — Mean: {group['time_end_trip_to_last_reminder_hrs'].mean():.2f} days  |  Median: {group['time_end_trip_to_last_reminder_hrs'].median():.2f} days")

In [ ]:
i = df[0].sort_values(['case:concept:name', 'time:timestamp'])

two_months_hrs = 60  # 60 days in hours

sequences = {
    'End trip → Declaration submitted': [],
    'End trip → Permit submitted': [],
    'End trip → Send Reminder → Declaration submitted': []
}

for case_id, case_df in i.groupby('case:concept:name'):
    activities = case_df['concept:name'].tolist()
    timestamps = case_df['time:timestamp'].tolist()
    
    for j in range(len(activities)):
        if activities[j] == 'End trip':
            
            # End trip → Declaration submitted (direct, no reminder in between)
            for k in range(j+1, len(activities)):
                if activities[k] == 'Declaration SUBMITTED by EMPLOYEE':
                    between = activities[j+1:k]
                    if 'Send Reminder' not in between:
                        total = (timestamps[k] - timestamps[j]).total_seconds() / 3600 / 24
                        sequences['End trip → Declaration submitted'].append(total)
                    break
            
            # End trip → Permit submitted (direct)
            for k in range(j+1, len(activities)):
                if activities[k] == 'Permit SUBMITTED by EMPLOYEE':
                    total = (timestamps[k] - timestamps[j]).total_seconds() / 3600 / 24
                    sequences['End trip → Permit submitted'].append(total)
                    break
            
            # End trip → Send Reminder → Declaration submitted
            k = j + 1
            reminder_count = 0
            while k < len(activities) and activities[k] == 'Send Reminder':
                reminder_count += 1
                k += 1
            if reminder_count > 0 and k < len(activities) and activities[k] == 'Declaration SUBMITTED by EMPLOYEE':
                total = (timestamps[k] - timestamps[j]).total_seconds() / 3600 / 24
                sequences['End trip → Send Reminder → Declaration submitted'].append(total)

print("=== Activity Sequence Summary ===\n")
print(f"{'Activity Sequence':<50} {'# Cases':>8} {'Median (days)':>12} {'Mean (days)':>10} {'% > 2 months':>13}")
print("-" * 100)

for name, times in sequences.items():
    if times:
        s = pd.Series(times)
        n_cases = len(s)
        median = s.median()
        mean = s.mean()
        pct_over_2_months = (s > two_months_hrs).sum() / len(s) * 100
        print(f"{name:<50} {n_cases:>8} {median:>12.2f} {mean:>10.2f} {pct_over_2_months:>12.2f}%")

In [ ]:
for name, i in zip(['Permit application', 'Declaration', 'Request for payment'], df[1:]):
    print(f"\n{'='*60}")
    print(f"=== {name} ===")
    print(f"{'='*60}")
    
    i = i.sort_values(['case:concept:name', 'time:timestamp'])
    i['next_activity'] = i.groupby('case:concept:name')['concept:name'].shift(-1)
    i['waiting_time'] = i.groupby('case:concept:name')['time:timestamp'].diff().dt.total_seconds() / 3600

    # count unique outputs per activity
    unique_outputs = (i.groupby('concept:name')['next_activity']
                      .nunique()
                      .reset_index())
    unique_outputs.columns = ['activity', 'unique_outputs']

    # count cases going through each activity
    cases_through = (i.groupby('concept:name')['case:concept:name']
                     .nunique()
                     .reset_index())
    cases_through.columns = ['activity', 'cases_through']

    # waiting time per activity
    wait_stats = (i.groupby('concept:name')['waiting_time']
                  .agg(['mean', 'median'])
                  .reset_index())
    wait_stats.columns = ['activity', 'mean_wait_hrs', 'median_wait_hrs']

    # combine
    summary = (unique_outputs
               .merge(cases_through, on='activity')
               .merge(wait_stats, on='activity'))
    
    summary['outputs_per_case'] = (summary['unique_outputs'] / summary['cases_through']).round(4)

    summary = summary.sort_values('outputs_per_case', ascending=False)

    print(summary.to_string(index=False))

In [ ]:
import pandas as pd

hours_per_day = 24

def get_path(group):
    return ' → '.join(group.sort_values('time:timestamp')['concept:name'].tolist())

summary_rows = []

for name, i in zip(['Complete process', 'Permit application', 'Declaration', 'Request for payment'], df):
    total_cases = i['case:concept:name'].nunique()
    unique_activities = i['concept:name'].nunique()

    # remove Start trip and End trip for time calculations
    i_time = i[~i['concept:name'].isin(['Start trip', 'End trip'])]

    # throughput time (excluding trip activities)
    throughput = i_time.groupby('case:concept:name')['time:timestamp'].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600 / hours_per_day)

    # loops (on full log)
    rework = i.groupby(['case:concept:name', 'concept:name']).size()
    rework_cases = rework[rework > 1].reset_index()
    loop_pct = rework_cases['case:concept:name'].nunique() / total_cases * 100

    # rejection rate (on full log)
    rejected = i[i['concept:name'].str.contains('REJECTED', case=False, na=False)]
    rejection_pct = rejected['case:concept:name'].nunique() / total_cases * 100

    # paths
    paths = i.groupby('case:concept:name').apply(get_path)
    path_counts = paths.value_counts()
    path_probs = path_counts / path_counts.sum()

    # unique paths
    n_unique_paths = len(path_counts)

    # top path concentration
    top_path_pct = path_probs.iloc[0] * 100

    # paths needed to cover 80% of cases
    cumulative = path_probs.cumsum()
    paths_for_80 = (cumulative < 0.80).sum() + 1

    summary_rows.append({
        'Subprocess': name,
        'Mean (days)': round(throughput.mean(), 2),
        'Median (days)': round(throughput.median(), 2),
        'Std deviation': round(throughput.std(), 2),
        '# unique activities': unique_activities,
        '% cases with loop': f"{loop_pct:.2f}%",
        'Rejection rate': f"{rejection_pct:.2f}%",
        '# unique paths': n_unique_paths,
        'Top path %': f"{top_path_pct:.2f}%",
        'Paths for 80% coverage': paths_for_80,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

In [ ]:
case_counts = []

for name, i in zip(['Complete process', 'Permit application', 'Declaration', 'Request for payment'], df):
    case_counts.append({
        'process': name,
        'num_cases': i['case:concept:name'].nunique(),
        'num_activities': i['concept:name'].nunique(),
        'num_events': len(i)
    })

case_counts_df = pd.DataFrame(case_counts)
print(case_counts_df.to_string(index=False))